# Demo — Auditing a CMS eCQM Regulatory Metric to Source
Veridian Health submits quarterly **CMS eCQM Timely Diagnostic Care** disclosures (`gold_cms_timely_care`) computed from `silver_clinical_encounter` and `silver_lab_result`. When the disclosed average wait time jumps from ~50 minutes to over 150 minutes, we use `lh.run_cms_quality_report()` to reconcile the Gold disclosure against `per_encounter` source records.

## Step 0: Connect to your Cloud Lab

Run the cell below to configure your AWS Cloud Lab environment and seed the **Veridian Health Medallion Lakehouse** (`veridian_lakehouse` in AWS Glue, S3 `bronze/silver/gold` Parquet prefixes, and AWS Lake Formation enforcement) via `setup_aws.py`.

In [1]:
import os
os.environ["VERIDIAN_LF_LIVE"] = "1"
os.environ["VERIDIAN_AWS_REGION"] = "us-east-1"
# If your Vocareum AWS credentials are not already in the environment, paste them below:
# os.environ["AWS_ACCESS_KEY_ID"] = "..."
# os.environ["AWS_SECRET_ACCESS_KEY"] = "..."
# os.environ["AWS_SESSION_TOKEN"] = "..."

import boto3
import setup_aws
glue = boto3.client("glue", region_name=os.environ["VERIDIAN_AWS_REGION"])
try:
    glue.get_database(Name="veridian_lakehouse")
    print("Glue database 'veridian_lakehouse' is already seeded and ready.")
except glue.exceptions.EntityNotFoundException:
    setup_aws.main()

Glue database 'veridian_lakehouse' is already seeded and ready.


In [2]:
import pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
from demo_toolkit import connect_lakehouse, lh_backend

lh = connect_lakehouse()
rep = lh.run_cms_quality_report()
print(f"Backend: {lh_backend()} -> {type(lh).__name__}")
display(rep.reconciliation)

Backend: live -> AwsMedallionLakehouse


  quarter  disclosed_total_min  disclosed_avg_min  lab_panels  recomputed_total_min  recomputed_avg_min  delta_total_min  matches
0  2023Q4               8078.3              155.4          52                2597.2                49.9           5481.1    False
1  2024Q1              34015.4              168.4         202               10380.0                51.4          23635.4    False
2  2024Q2              38305.7              186.0         206               10935.6                53.1          27370.1    False
3  2024Q3              33401.2              155.4         215               10934.0                50.9          22467.2    False
4  2024Q4              26612.1              141.6         188                9360.5                49.8          17251.6    False


## 1. Inspect the quantile distribution of `turnaround_min` across all lab panels

In [3]:
print("Total encounter lab panels:", len(rep.per_encounter))
print("\nQuantiles of turnaround_min:")
print(rep.per_encounter["turnaround_min"].quantile([0.5, 0.9, 0.95, 0.99, 1.0]).round(1).to_string())
print("\nSample panels with turnaround_min > 240 (recorded in seconds instead of minutes):")
display(rep.per_encounter[rep.per_encounter["turnaround_min"] > 240].head(8))

Total encounter lab panels: 863

Quantiles of turnaround_min:
0.50      52.6
0.90      89.1
0.95      94.2
0.99    4196.5
1.00    5574.0

Sample panels with turnaround_min > 240 (recorded in seconds instead of minutes):
     encounter_id         encounter_ts acuity_level  result_id loinc_code                     panel_name  turnaround_min         completed_ts quarter
13           5017  2024-01-25T00:00:00      routine       9014    24323-8  Comprehensive Metabolic Panel          5046.0  2024-01-25T02:00:00  2024Q1
16           5020  2024-01-15T21:00:00      routine       9017    11502-2      High-Sensitivity Troponin          2508.0  2024-01-15T23:00:00  2024Q1
29           5033  2024-11-05T16:00:00      routine       9030    14647-2                    Lipid Panel          1230.0  2024-11-05T18:00:00  2024Q4
32           5036  2024-07-08T23:00:00    follow_up       9033    11502-2      High-Sensitivity Troponin          5190.0  2024-07-09T01:00:00  2024Q3
53           5057  2024-01-16T

**Root cause:** Through the 95th percentile (`94.2 min`), lab turnaround times sit between `8` and `95` minutes. Above p95 (`32` of `863` lab panels, ~3.7%), an analyzer feed emits turnaround time in **seconds** (`60x` scale, `480..5,574`), inflating every quarterly CMS eCQM disclosure roughly 3x until normalized at the Silver -> Gold boundary.